
# <div align="center">**CUSTOMER CHURN PREDICTION**</div>  

### **MỰC TIÊU DỰ ÁN**

- **Giảm tỷ lệ khách hàng rời bỏ (churn)**: Chủ động dự đoán khách hàng có khả năng rời bỏ sớm, từ đó đưa ra các chính sách giữ chân kịp thời.

- **Tối ưu chi phí giữ chân khách hàng:** Thay vì dàn trải nguồn lực, ngân hàng tập trung chính xác vào nhóm khách hàng có nguy cơ cao nhất.

- **Nâng cao trải nghiệm khách hàng:** Hiểu rõ yếu tố nào dẫn đến khách hàng rời bỏ, từ đó cải thiện dịch vụ, sản phẩm một cách chính xác và hiệu quả nhất.

- **Gia tăng lợi thế cạnh tranh:** Sử dụng công nghệ (AI/ML) để nhanh chóng phát hiện và phản ứng kịp thời trước biến động hành vi khách hàng, từ đó chiếm ưu thế trước các đối thủ cạnh tranh trên thị trường.
  
### **DỮ LIỆU ĐẦU VÀO**
* **Thông tin khách hàng**

  * Định danh: CustomerId

  * Nhân khẩu học: Gender, Age

  * Quan hệ với ngân hàng: Tenure (số năm gắn bó)

* **Hoạt động tài chính**

  * Điểm tín dụng: CreditScore

  * Số dư tài khoản: Balance

  * Thu nhập ước tính: EstimatedSalary

  * Sản phẩm đang sử dụng: NumOfProducts, HasCrCard

* **Hành vi sử dụng dịch vụ**

  * Tần suất đăng nhập: LoginFrequency

  * Giao dịch ra ngoài hệ thống: ExternalTransfers

  * Tình trạng hoạt động: IsActiveMember

* **Biến mục tiêu (Target)**

  * Rời bỏ dịch vụ: Exited (1 = đã rời, 0 = chưa)
      
### **NỘI DUNG CHÍNH**
* **Thiết lập dự án**

* **Tiền xử lý & Feature Engineering**


* **EDA cơ bản**


* **Xây mô hình dự đoán churn**


* **Giải thích mô hình với SHAP**

* **Tạo khoảng tin cậy với MAPIE**


---
# **THIẾT LẬP DỰ ÁN**
---

# **0. Cài đặt thư viện**
> ⚠️ Chạy cell này trước toàn bộ notebook để tránh lỗi kernel restart giữa chừng.

In [ ]:
!pip install shap mapie xgboost lightgbm -q


# **1. Import thư viện & cấu hình toàn cục**

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import joblib
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve
)
from sklearn.feature_selection import mutual_info_classif

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import shap
from mapie.classification import MapieClassifier
from mapie.metrics import (
    classification_coverage_score,
    classification_mean_width_score
)

# ── Hằng số dùng xuyên suốt notebook ──
SEED = 42
TEST_SIZE   = 0.20   # 20% test
CALIB_SIZE  = 0.125  # 12.5% của phần còn lại → 10% tổng
ALPHA_MAPIE = 0.05   # Độ tin cậy 95%

print('✅ Import hoàn tất')


# **2. Load dữ liệu**

In [ ]:
df = pd.read_excel("/content/BankingDataset.xlsx")
df.head()


# **3. Kiểm tra định dạng dữ liệu, shape, missing value**

In [ ]:
print("Shape:", df.shape)
df.info()


In [ ]:
missing_ratio = df.isnull().mean().sort_values(ascending=False)
print("Cột có giá trị thiếu:")
display(missing_ratio[missing_ratio > 0])


---
# **TIỀN XỬ LÝ & FEATURE ENGINEERING**
---

# **1. Mapping biến phân loại ( Gender )**

In [ ]:
gender_map = {"Male": 1, "Female": 0}
df["Gender"] = df["Gender"].map(gender_map)

os.makedirs("models", exist_ok=True)
with open("models/gender_map.json", "w") as f:
    json.dump(gender_map, f)

print("✅ Mapping Gender xong")




# **2. Feature Engineering**

Tạo ra các biến mới từ các biến gốc để mô hình có thêm "góc nhìn" mới về hành vi khách hàng :
 - Ý tưởng: Nếu khách có số dư cao nhưng chỉ dùng ít sản phẩm → tiềm năng chưa khai thác hết → có thể là dấu hiệu churn.
 - So sánh giữa thời gian khách hàng gắn bó và tần suất tương tác → khách gắn bó lâu nhưng ít login → có nguy cơ bỏ đi.
 -  Một khách hàng trẻ nhưng có thu nhập cao có thể được ưu tiên hơn → mô hình học được sự khác biệt giữa "khách VIP tiềm năng" và khách thường.
 - Login nhiều nhưng ít chuyển khoản → có thể chỉ truy cập xem, không thật sự tương tác tài chính.
  
  Dùng để phân loại hành vi "ngó chơi" vs "giao dịch thật".



In [ ]:
df["BalancePerProduct"]  = df["Balance"] / (df["NumOfProducts"] + 1)
df["ActivityPerTenure"]  = df["LoginFrequency"] / (df["Tenure"] + 1)
df["SalaryPerAge"]       = df["EstimatedSalary"] / (df["Age"] + 1)
df["TransferLoginRatio"] = df["ExternalTransfers"] / (df["LoginFrequency"] + 1)

print("✅ Feature Engineering xong. Shape mới:", df.shape)
df.head(3)


---
# **EDA CƠ BẢN**
---

# **1. Check thông số và phân vị các cột để biết có cần lọc outlier hay không**

In [ ]:
df.describe()



# **2. Các cột được lựa chọn để xử lý outlier**

> **FIX:** Mở rộng xử lý outlier sang cả các cột gốc quan trọng (`Age`, `CreditScore`, `Balance`, `EstimatedSalary`) thay vì chỉ xử lý các feature engineered.


In [ ]:
# Mở rộng: xử lý cả cột gốc lẫn feature engineered
cols_to_clip = [
    "Age", "CreditScore", "Balance", "EstimatedSalary",
    "BalancePerProduct", "ActivityPerTenure",
    "SalaryPerAge", "TransferLoginRatio"
]

outlier_log = {}

for col in cols_to_clip:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    before = df[col].copy()
    df[col] = df[col].clip(lower, upper)

    clipped = ((before < lower) | (before > upper)).sum()
    outlier_log[col] = clipped

print("Kết quả xử lý outlier (IQR clip):")
for col, clipped in sorted(outlier_log.items(), key=lambda x: x[1], reverse=True):
    print(f"  {col:30s} → {clipped} dòng bị clip")


# **3. Vẽ biểu đồ phân phối để quan sát dữ liệu sau khi cắt outlier**

In [ ]:
numeric_cols = df.select_dtypes(include=['number']).columns

plt.figure(figsize=(20, len(numeric_cols) * 2.5))
for i, col in enumerate(numeric_cols, 1):
    plt.subplot(len(numeric_cols), 3, i)
    sns.histplot(data=df, x=col, bins=30, kde=True, color='skyblue')
    plt.title(f"Phân phối: {col}", fontsize=10)
    plt.xlabel("")
    plt.tight_layout()

plt.suptitle("Biểu đồ phân phối các biến số", fontsize=14, y=1.02)
plt.show()



# **4. Kiểm tra tương quan giữa các biến**

 - Dùng ma trận tương quan để kiểm tra và lọc các biến có độ tương quan quá lớn với nhau
 - Mutual Information (Thông tin tương hỗ) : Để check tương quan giữa các biến với đầu ra ( Exited )


In [ ]:
corr_matrix = df.corr(numeric_only=True)

plt.figure(figsize=(14, 12))
sns.heatmap(corr_matrix,
            annot=True,
            fmt=".2f",
            cmap="coolwarm",
            center=0,
            linewidths=0.5,
            cbar_kws={"shrink": 0.8})
plt.title("Correlation Matrix - Full Features (Pearson)", fontsize=16)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()




---


# **CHURN PREDICTION MODEL**




---


# **1. Chọn các biến đầu vào dựa theo Ma trận tương quan và Mutual Information**


In [ ]:
selected_features = [
    "ExternalTransfers",
    "TransferLoginRatio",
    "NumOfProducts",
    "Age",
    "IsActiveMember",
    "BalancePerProduct",
    "ActivityPerTenure",
    "SalaryPerAge"
]





# **2. Chia dữ liệu huấn luyện**

  ### **Tách test set trước – 20%**

    - X_test, y_test sẽ là tập test cuối cùng – chỉ dùng sau cùng để đánh giá mô hình đã huấn luyện.

    - Dùng stratify=y để đảm bảo tỉ lệ churn (Exited = 1) trong tập test giống với toàn bộ tập dữ liệu, tránh bị lệch.

    - random_state=SEED để tái lập kết quả.

   ### **Tách Calibration set – 10% của tổng, tương đương 12.5% phần còn lại**

    - X_calib, y_calib: Tập calibration – dùng để tạo ra khoảng tin cậy trong dự đoán (sử dụng thư viện MAPIE).

    - X_train, y_train: Tập huấn luyện chính.

    - Vì tách 12.5% của phần còn lại (80% dữ liệu ban đầu), nên:

      - X_train = 70%

      - X_calib = 10%

      - X_test = 20%



In [ ]:
X = df[selected_features]
y = df["Exited"]

# 1. Tách test trước (20%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED
)

# 2. Tách calib từ phần còn lại (10% tổng → 12.5% của X_temp)
X_train, X_calib, y_train, y_calib = train_test_split(
    X_temp, y_temp, test_size=CALIB_SIZE, stratify=y_temp, random_state=SEED
)

print(f"X_train : {X_train.shape}  ({len(X_train)/len(X)*100:.0f}%)")
print(f"X_calib : {X_calib.shape}  ({len(X_calib)/len(X)*100:.0f}%)")
print(f"X_test  : {X_test.shape}  ({len(X_test)/len(X)*100:.0f}%)")



# **3. Tính Mutual Information trên tập train**



In [ ]:
mi_scores = mutual_info_classif(
    X_train.fillna(0), y_train, discrete_features='auto', random_state=SEED
)
mi_series = pd.Series(mi_scores, index=X_train.columns).sort_values(ascending=False)

print("Mutual Information scores (tính trên X_train):")
print(mi_series)

mi_series.plot(kind='barh', figsize=(10, 6))
plt.title("Mutual Information with 'Exited' (train set only)")
plt.xlabel("MI Score")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


# **4. Machine Learning**

### **Trong dự án này, tôi đã triển khai 3 mô hình classification chính để dự đoán churn:**

- Random Forest (RF)

- XGBoost

- LightGBM

Các mô hình này:

 - Không cần scale dữ liệu

 - Xử lý tốt các mối quan hệ phi tuyến

 - Chịu được noise và feature dư thừa

Ngoài ra:

 - Với dữ liệu mất cân bằng (Exited = 1 chỉ ~20%), đã sử dụng tham số:

 - class_weight='balanced' cho RF và LGBM

 - scale_pos_weight cho XGBoost


### **Random Forest**

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=500,
    random_state=SEED,
    class_weight='balanced'
)
rf_model.fit(X_train, y_train)
print(" Random Forest trained")


In [ ]:
y_pred_rf = rf_model.predict(X_test)
print("Đánh giá Random Forest:")
print(classification_report(y_test, y_pred_rf, digits=4))


### **XGBoost**

In [ ]:
scale_weight = y_train.value_counts()[0] / y_train.value_counts()[1]

xgb_model = XGBClassifier(
    n_estimators=500,
    scale_pos_weight=scale_weight,
    random_state=SEED,
    eval_metric='logloss'
)
xgb_model.fit(X_train, y_train)
print(" XGBoost trained")


In [ ]:
y_pred_xgb = xgb_model.predict(X_test)
print("Đánh giá XGBoost:")
print(classification_report(y_test, y_pred_xgb, digits=4))


### **LightGBM**

In [ ]:
lgb_model = LGBMClassifier(
    n_estimators=500,
    class_weight='balanced',
    random_state=SEED
)
lgb_model.fit(X_train, y_train)
print(" LightGBM trained")


In [ ]:
y_pred_lgb = lgb_model.predict(X_test)
print("Đánh giá LightGBM:")
print(classification_report(y_test, y_pred_lgb, digits=4))


# **4. Thông số đánh giá**
### **Accuracy (Độ chính xác tổng thể)**

$$
\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}
$$

**Ý nghĩa:**  
- Tỷ lệ dự đoán đúng (churn và không churn) trên toàn bộ mẫu.

**Nhược điểm:**  
 - Không đáng tin khi dữ liệu mất cân bằng.  
**VD:** *nếu chỉ 20% khách hàng churn, đoán tất cả đều không churn cũng được 80% accuracy → tưởng tốt nhưng thực ra tệ*.

**Khi nào dùng:**  
 - Chỉ dùng để tham khảo chung. Không nên là chỉ số chính khi tỷ lệ churn < 50%.

<br>

### **Precision (Độ chính xác dương tính)**

$$
\text{Precision} = \frac{TP}{TP + FP}
$$

**Ý nghĩa:**  
- Trong số khách hàng được **dự đoán sẽ rời bỏ**, bao nhiêu người **thực sự sẽ rời**?

**Ví dụ:**  
- Nếu Precision = 0.85 → trong 100 khách bị báo là sẽ churn, có 85 người thật sự sẽ đi.

**Khi nào dùng:**  
- Quan trọng khi chi phí xử lý khách nhầm cao.  
- Nhưng nếu Precision cao mà Recall thấp thì mô hình "chơi an toàn", chỉ báo churn khi rất chắc → bỏ sót nhiều khách thật sự rời đi.

<br>

### **Recall (Độ nhạy)**

$$
\text{Recall} = \frac{TP}{TP + FN}
$$

**Ý nghĩa:**  
- Trong số khách hàng **thực sự rời bỏ**, mô hình **bắt được bao nhiêu**?

**Ví dụ:**  
- Nếu Recall = 0.90 → có 100 người thật sự sẽ rời, mô hình bắt được 90 người.

**Khi nào dùng:**  
 Rất quan trọng trong dự án churn vì:
- Bỏ sót khách sắp rời là **mất tiền thật**.
- Dự đoán sai còn xử lý được (ví dụ gọi điện hỏi thăm), nhưng bỏ qua thì mất luôn.

<br>

### **F1-Score (Chỉ số cân bằng Precision và Recall)**

$$
\text{F1} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}
$$

**Ý nghĩa:**  
F1-score là chỉ số tổng hợp giữa độ chính xác (Precision) và độ nhạy (Recall).  
Nếu một trong hai thấp, F1 sẽ phản ánh điều đó. F1-score là thước đo tốt khi cần cân bằng giữa việc tránh bỏ sót và tránh báo nhầm.

**Khi nào sử dụng:**  
- Khi dự án yêu cầu mô hình phải bắt được đúng khách hàng rời bỏ nhưng cũng hạn chế báo nhầm.  
- Thích hợp với các bài toán như churn prediction, fraud detection, medical diagnosis.

**Sau khi tính Precision, Recall, F1-score cho từng lớp (class 0: không rời, class 1: rời), ta cần tổng hợp kết quả chung. sklearn cung cấp hai dạng trung bình:**

<br>

### **Macro Average**

**Cách tính:**  
Tính trung bình cộng của chỉ số trên tất cả các lớp mà **không xét đến tỉ lệ số lượng** giữa các lớp.

Ví dụ cho F1-score:

$$
\text{F1}_{macro} = \frac{\text{F1}_{class\ 0} + \text{F1}_{class\ 1}}{2}
$$

**Ý nghĩa:**  
- Đánh giá mô hình công bằng giữa các lớp.
- Phù hợp khi cần mô hình thể hiện tốt cho **cả hai lớp**, dù dữ liệu mất cân bằng.

<br>

### **Weighted Average**

**Cách tính:**  
Tính trung bình cộng có trọng số, trọng số là số lượng mẫu tương ứng ở mỗi lớp.

Ví dụ cho F1-score:

$$
\text{F1}_{weighted} = \frac{\text{F1}_0 \cdot N_0 + \text{F1}_1 \cdot N_1}{N_0 + N_1}
$$

**Ý nghĩa:**  
- Phản ánh đúng hiệu suất tổng thể của mô hình trên toàn bộ dữ liệu.
- Tuy nhiên có thể **che khuất điểm yếu của mô hình ở lớp thiểu số**, do lớp đông hơn chiếm nhiều trọng số.

<br>

### **Lưu ý khi áp dụng cho bài toán churn prediction:**

- `Recall` và `F1-score` của lớp **Exited = 1** (khách hàng rời) là chỉ số quan trọng nhất.
- `Macro average` giúp đánh giá xem mô hình có đang thiên lệch không.
- `Weighted average` chỉ nên dùng để đánh giá tổng thể, không phản ánh chính xác hiệu quả của mô hình trên lớp churn (nếu mất cân bằng).



# **5. So sánh mô hình**

- Dựa vào bộ chỉ số đầu ra và ma trận nhầm lẫn của các mô hình ta có thể so sánh và chọn ra mô hình sẽ tích hợp SHAP và MAPIE để đưa vào hệ thống sau cùng.

> **FIX:** Bổ sung ROC-AUC và PR-AUC — hai metric quan trọng nhất cho bài toán imbalanced classification.


In [ ]:
models = {
    "Random Forest": (rf_model, y_pred_rf),
    "XGBoost":       (xgb_model, y_pred_xgb),
    "LightGBM":      (lgb_model, y_pred_lgb),
}

# ── Confusion Matrices ──
fig, axs = plt.subplots(1, 3, figsize=(16, 5))
cmaps = ['Blues', 'Greens', 'Oranges']

for ax, (name, (model, y_pred)), cmap in zip(axs, models.items(), cmaps):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax)
    ax.set_title(f"Confusion Matrix - {name}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()


In [ ]:
# ── ROC & PR Curves ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

summary_rows = []

for name, (model, y_pred) in models.items():
    proba = model.predict_proba(X_test)[:, 1]

    # ROC
    fpr, tpr, _ = roc_curve(y_test, proba)
    roc_auc = roc_auc_score(y_test, proba)
    ax1.plot(fpr, tpr, label=f"{name} (AUC={roc_auc:.4f})")

    # PR
    prec, rec, _ = precision_recall_curve(y_test, proba)
    pr_auc = average_precision_score(y_test, proba)
    ax2.plot(rec, prec, label=f"{name} (PR-AUC={pr_auc:.4f})")

    report = classification_report(y_test, y_pred, output_dict=True)
    summary_rows.append({
        "Model": name,
        "ROC-AUC": round(roc_auc, 4),
        "PR-AUC":  round(pr_auc, 4),
        "Precision (class 1)": round(report['1']['precision'], 4),
        "Recall (class 1)":    round(report['1']['recall'], 4),
        "F1 (class 1)":        round(report['1']['f1-score'], 4),
        "Macro F1":            round(report['macro avg']['f1-score'], 4),
    })

ax1.plot([0, 1], [0, 1], 'k--')
ax1.set_xlabel("FPR"); ax1.set_ylabel("TPR")
ax1.set_title("ROC Curve"); ax1.legend()

ax2.set_xlabel("Recall"); ax2.set_ylabel("Precision")
ax2.set_title("Precision-Recall Curve"); ax2.legend()

plt.tight_layout()
plt.show()

summary_df = pd.DataFrame(summary_rows).set_index("Model")
display(summary_df)


# **6. Chọn mô hình tốt nhất & lưu**

> **FIX:** Lưu model **sau** khi so sánh, và chỉ lưu model được chọn để dùng cho SHAP + MAPIE.  
> Tiêu chí chọn: **ROC-AUC cao nhất** (phù hợp với dữ liệu mất cân bằng).


In [ ]:
# Chọn model tốt nhất theo ROC-AUC
best_name = summary_df["ROC-AUC"].idxmax()
best_model_map = {
    "Random Forest": rf_model,
    "XGBoost":       xgb_model,
    "LightGBM":      lgb_model,
}
best_model = best_model_map[best_name]
print(f" Model tốt nhất: {best_name}  (ROC-AUC={summary_df.loc[best_name,'ROC-AUC']})")

# Lưu tất cả model và feature list
os.makedirs("models", exist_ok=True)
joblib.dump(rf_model,  "models/rf_model_final.pkl")
joblib.dump(xgb_model, "models/xgb_model_final.pkl")
joblib.dump(lgb_model, "models/lgb_model_final.pkl")

with open("models/feature_list.json", "w") as f:
    json.dump(list(X_train.columns), f)

with open("models/best_model_name.txt", "w") as f:
    f.write(best_name)

print(" Đã lưu tất cả models vào thư mục models/")


---
# **GIẢI THÍCH MÔ HÌNH VỚI SHAP**
---

# **SHAP là gì và tại sao cần dùng?**
### **SHAP (SHapley Additive exPlanations) là một phương pháp giúp giải thích đầu ra của mô hình Machine Learning**
 - Nó trả lời câu hỏi: "Tại sao mô hình dự đoán khách hàng này sẽ rời bỏ?"

 - Thay vì chỉ biết một con số xác suất churn, SHAP giúp chúng ta thấy được các yếu tố nào đang kéo xác suất đó lên hoặc xuống, giúp mô hình trở nên minh bạch và dễ tin hơn với người dùng kinh doanh.

### **Ứng dụng trong dự án Churn Prediction**
Trong dự án này, chúng tôi áp dụng SHAP để:

 - Giải thích các yếu tố ảnh hưởng đến từng quyết định dự đoán churn.

 - Trực quan hóa vai trò của từng biến đầu vào.

 - Tạo sự minh bạch, tăng độ tin tưởng từ phía khách hàng và phòng kinh doanh.



In [ ]:
# Load feature list để đảm bảo đúng thứ tự cột
with open("models/feature_list.json") as f:
    feature_list = json.load(f)

X_test_shap = X_test[feature_list]

# Tạo explainer cho tree model (dùng best_model)
explainer = shap.TreeExplainer(best_model)
shap_values_raw = explainer.shap_values(X_test_shap)

# FIX: Handle cả binary (2D) lẫn multiclass (3D) output của SHAP
# LightGBM binary trả về list 2 phần tử → lấy index 1 (class=churn)
if isinstance(shap_values_raw, list):
    shap_values = shap_values_raw[1]
    expected_value = explainer.expected_value[1]
else:
    shap_values = shap_values_raw
    expected_value = explainer.expected_value

print(f"SHAP values shape : {shap_values.shape}")
print(f"Expected value    : {expected_value:.4f}")





# **SHAP Bar Plot (Feature Importance tổng thể)**


- Trung bình độ ảnh hưởng của mỗi biến đến dự đoán trên toàn bộ khách hàng.

  - Từ biểu đồ ta thấy ExternalTransfers có tác động lớn nhất đến mô hình.

  - Tiếp theo là BalancePerProduct, SalaryPerAge, Age, v.v.

- Ý nghĩa:
  - Giúp team kinh doanh hiểu biến nào đang thực sự quan trọng với mô hình – từ đó tập trung cải thiện/thu thập thêm thông tin cho các biến này.


<br>


# **SHAP Summary Plot (Violin plot)**



- Kết hợp giữa giá trị biến và mức độ ảnh hưởng để thấy xu hướng hành vi toàn bộ hệ thống.


- Trục ngang: SHAP value (mức độ ảnh hưởng).

- Màu sắc: Giá trị của feature (Xanh: thấp, Hồng: cao).

- Ví dụ phân tích:

  - Với ExternalTransfers: giá trị thấp (xanh) thường kéo SHAP về âm → khả năng rời đi tăng cao khi ít giao dịch ra ngoài.

  - Với BalancePerProduct: giá trị thấp cũng thường đẩy dự đoán lên → tức là khách có tài khoản nhưng không dùng sản phẩm có khả năng rời cao.

- Ý nghĩa:
  - Cho thấy mối quan hệ logic giữa hành vi và churn trên toàn hệ thống. Điều này giúp validate mô hình, đảm bảo mô hình "nghĩ đúng".



In [ ]:
shap.summary_plot(shap_values, X_test_shap, plot_type="bar")
shap.summary_plot(shap_values, X_test_shap)





# **SHAP Force Barplot**

 Force plot (SHAP Decision Explanation)
 - Mỗi biểu đồ là một cá nhân – mô hình đã quyết định họ rời hay không, và SHAP cho thấy lý do.

  - Màu đỏ: Các biến làm tăng khả năng rời bỏ.

  - Màu xanh: Các biến làm giảm khả năng rời bỏ.

Trục ngang thể hiện mức độ ảnh hưởng (âm hay dương) tới đầu ra mô hình.

- Ví dụ giải thích:

  - ExternalTransfers = 13 làm giảm mạnh khả năng rời bỏ → khách này giao dịch ra ngoài nhiều, có thể vẫn đang tương tác tích cực.

  - BalancePerProduct = 0 làm tăng khả năng rời bỏ → khách có tài khoản nhưng không sử dụng sản phẩm → tiềm năng rời đi cao.

 ⟶ Giúp nhân viên CSKH hiểu lý do tại sao hệ thống đánh giá như vậy, từ đó hành động đúng hướng.



In [ ]:
# Waterfall plot – khách hàng #0
shap.plots.waterfall(shap.Explanation(
    values=shap_values[0],
    base_values=expected_value,
    data=X_test_shap.iloc[0],
    feature_names=feature_list
))


In [ ]:
# Waterfall plot – khách hàng #1
shap.plots.waterfall(shap.Explanation(
    values=shap_values[1],
    base_values=expected_value,
    data=X_test_shap.iloc[1],
    feature_names=feature_list
))



# **Giải thích các chỉ số trong SHAP Force Plot**




SHAP force plot là biểu đồ trực quan mô tả lý do tại sao mô hình đưa ra một dự đoán cụ thể cho từng khách hàng. Dưới đây là các thành phần chính trong biểu đồ:


### **1. `f(x)` – Giá trị đầu ra của mô hình**

$$
f(x) = \text{logit score}
$$

- Là giá trị đầu ra (log-odds) của mô hình cho một cá nhân cụ thể.
- Được dùng để tính xác suất bằng hàm sigmoid:

$$
\text{Probability} = \frac{1}{1 + e^{-f(x)}}
$$

- Nếu `f(x)` rất âm → xác suất churn thấp. Nếu `f(x)` dương → xác suất churn cao.

**Ví dụ:**  
`f(x) = -13.45` → xác suất churn gần bằng 0%.

<br>


### **2. `E[f(X)]` – Giá trị trung bình mô hình**

- Là giá trị đầu ra trung bình của mô hình trên toàn bộ tập dữ liệu huấn luyện.
- Là điểm khởi đầu cho mọi dự đoán trước khi thêm ảnh hưởng của từng biến.

**Ý nghĩa:**  
Nếu không có thông tin gì về khách hàng, mô hình mặc định dự đoán bằng `E[f(X)]`.

**Ví dụ:**  
`E[f(X)] = -5.625` → baseline của toàn bộ hệ thống.

<br>

### **3. Các mũi tên (các biến đầu vào)**

- **Màu đỏ:** Biến này đang **làm tăng xác suất churn**.
- **Màu xanh:** Biến này đang **giảm xác suất churn**.
- **Chiều dài mũi tên:** Mức độ ảnh hưởng của biến đó.

**Ví dụ:**  
- `ExternalTransfers = 13 → -8.9`  
Khách chuyển tiền ra ngoài nhiều → được coi là đang hoạt động tích cực → giảm mạnh khả năng churn.

- `BalancePerProduct = 0 → +1.86`  
Khách có tài khoản nhưng không dùng sản phẩm → khả năng churn tăng.

<br>

### **4. Tổng hợp: Công thức SHAP**

Biểu đồ force plot hoạt động theo công thức:

$$
f(x) = E[f(X)] + \sum_{i=1}^{n} \text{SHAP}_i
$$

Tức là: mô hình bắt đầu từ điểm trung bình và cộng/trừ ảnh hưởng của từng biến để đi đến quyết định cuối cùng.





---


# **TẠO KHOẢNG TIN CẬY VỚI MAPIE**

---






# **Giới thiệu về MAPIE**




**MAPIE** (Model Agnostic Prediction Interval Estimator) là một thư viện Python hỗ trợ tạo **khoảng dự đoán (prediction intervals)** hoặc **tập dự đoán (prediction sets)** cho các mô hình học máy, dựa trên phương pháp conformal prediction.

Khác với mô hình truyền thống chỉ đưa ra một kết quả duy nhất, MAPIE cho phép dự đoán **nhiều khả năng xảy ra**, cùng với **mức độ tin cậy xác định trước** (ví dụ: 90%, 95%).

MAPIE hỗ trợ cả:
- Phân loại (Classification)
- Hồi quy (Regression)



# **Cách hoạt động của MAPIE**

MAPIE hoạt động theo các bước sau:

1. **Tách một phần dữ liệu làm calibration set (tập hiệu chỉnh)**.
2. **Huấn luyện mô hình như bình thường trên phần còn lại**.
3. **Tính toán sai số dự đoán hoặc xác suất trên tập calibration**.
4. **Sinh ra khoảng/tập dự đoán dựa trên phân bố sai số và tham số `alpha` do người dùng cung cấp**.


# **Các tham số quan trọng trong MAPIE**

### **Alpha**

- Đại diện cho mức độ sai số cho phép.
- Công thức:

  ```
  alpha = 1 - độ tin cậy mong muốn
  ```

- Ví dụ: Nếu yêu cầu độ tin cậy 95%, thì:

  ```
  alpha = 1 - 0.95 = 0.05
  ```


### **Coverage**

- Là tỷ lệ các điểm test mà prediction interval (hoặc prediction set) **bao phủ đúng giá trị thực tế**.
- Dùng để đánh giá chất lượng dự đoán của mô hình sau khi áp dụng MAPIE.


# **Vai trò trong dự án**

MAPIE không thay thế mô hình gốc.  
Thay vào đó, nó hoạt động như một **lớp bổ sung** giúp:

- Tăng độ tin cậy cho mô hình dự đoán.
- Cung cấp thông tin xác suất và mức độ chắc chắn.
- Hỗ trợ người dùng ra quyết định tốt hơn.

Ví dụ: hệ thống không chỉ dự đoán "khách hàng này sẽ churn", mà còn đưa ra khoảng tin cậy: "Dự đoán này đúng với xác suất 90%".



# **Các bước triển khai:**

1. **Chia dữ liệu thành 3 phần:**  
   - Train (70%)  
   - Calibration (10%)  
   - Test (20%)

   Khác với thông thường chỉ chia Train/Test.

2. **Huấn luyện mô hình trên tập Train.**

3. **Áp dụng MAPIE trên tập Calibration:**
   - Tính toán xác suất phân loại cho từng mẫu.
   - Ước lượng ngưỡng để prediction set đạt coverage thực tế ≥ (1 - alpha).

4. **Dự đoán trên tập Test:**
   - Trả về prediction set với độ tin cậy cụ thể (ví dụ: 95%).


In [ ]:
# Fit MAPIE với best_model đã huấn luyện (cv='prefit' vì model đã train rồi)
mapie_clf = MapieClassifier(estimator=best_model, method="lac", cv="prefit")
mapie_clf.fit(X_calib, y_calib)
print(" MAPIE calibration xong")


In [ ]:
# ── Khảo sát coverage & width theo nhiều alpha ──
alpha_range = np.arange(0.01, 0.98, 0.01)
_, y_ps_score = mapie_clf.predict(X_test, alpha=alpha_range)

coverages = [
    classification_coverage_score(y_test, y_ps_score[:, :, i])
    for i in range(len(alpha_range))
]
widths = [
    classification_mean_width_score(y_ps_score[:, :, i])
    for i in range(len(alpha_range))
]

fig, axs = plt.subplots(1, 2, figsize=(12, 5))

axs[0].scatter(1 - alpha_range, coverages, s=8, label="lac")
axs[0].plot([0, 1], [0, 1], 'k--', label="x=y")
axs[0].set_xlabel("1 - alpha (độ tin cậy yêu cầu)")
axs[0].set_ylabel("Coverage thực tế")
axs[0].set_title("Coverage vs. Confidence level")
axs[0].legend()

axs[1].scatter(1 - alpha_range, widths, s=8, label="lac")
axs[1].set_xlabel("1 - alpha")
axs[1].set_ylabel("Average prediction set size")
axs[1].set_title("Width vs. Confidence level")
axs[1].legend()

plt.tight_layout()
plt.show()

results_df = pd.DataFrame({
    'alpha':     alpha_range,
    'coverage':  coverages,
    'width':     widths,
})
display(results_df.head(20))




- **Mục đích của biểu đồ 1** : Đo mức độ mà prediction set thực tế bao phủ đúng nhãn thật so với độ tin cậy kỳ vọng của người dùng.

  - Mô hình được calibrate rất tốt: coverage thực tế ≈ độ tin cậy đặt ra.
  - Không bị quá tự tin (coverage < kỳ vọng) hay quá dè chừng (coverage >> kỳ vọng).
  - Đây là dấu hiệu mô hình có thể được dùng an toàn trong hệ thống thực tế mà không cần tuning thêm.



- **Mục đích của biểu đồ 2** này giúp ta đo mức độ "rộng" của tập dự đoán khi ta yêu cầu mô hình có độ tin cậy cao hơn.
  - Prediction set có kích thước trung bình tăng tuyến tính và hợp lý theo yêu cầu độ tin cậy.
  - Ở độ tin cậy thấp khiến prediction set nhỏ (≈ 1 nhãn), phù hợp để tự động hóa
  - Ở độ tin cậy cao dẫn đến prediction set mở rộng từ từ và đảm bảo không bỏ sót nhưng vẫn giữ tính cụ thể cao

  - Cho thấy mô hình rất "tự tin khi nên" và "cẩn thận khi cần" → cực kỳ thích hợp để đưa vào các pipeline có yếu tố quyết định tự động + kiểm soát rủi ro.

- Tiếp đến chúng tôi quan sát bảng thống kê thông số của alpha, coverage, width để chọn ra alpha phù hợp nhất. Mục tiêu là cân bằng giữa coverage với width theo alpha

  - Tức là coverages sẽ sát với 1-alpha (độ tin cậy). Lúc này mô hình đạt đúng mức độ tin cậy kỳ vọng. Không overconfident (nguy hiểm), cũng không quá dè chừng (thiếu hiệu quả).
Width càng gần 1 càng tốt. Vì trong trường hợp Width = 1 thì Prediction set chỉ chứa đúng 1 nhãn duy nhất, và nhãn đó chính là nhãn đúng.

  - Để tìm ra điểm trade-off hợp lí. Tức là điểm mà mô hình dự đoán đúng nhiều nhất và trả lại ít hồ sơ cần người dùng phải tự xem xét nhất có thể tức là nó trả về nhiều set có nhiều giá trị.



# **MAPIE tính cận trên và dưới như thế nào?**

### **Bước 1: Tính xác suất trên tập calibration (`X_cal`)**

Gọi:
- ^Pi: xác suất dự đoán class 1 của mẫu \( i \)  
- \( y_i \): nhãn thật của mẫu \( i \)

<br>

### **Bước 2: Tính conformity score**

$$
s_i =
\begin{cases}
1 - \hat{p}_i, & \text{nếu } y_i = 1 \\
\hat{p}_i, & \text{nếu } y_i = 0
\end{cases}
$$

Score này cho biết mô hình đã sai lệch bao nhiêu so với thực tế.

<br>

### **Bước 3: Tính quantile tương ứng với alpha**

Với alpha = 0.05, ta tính:

$$
q_\alpha = \text{Quantile}_{1 - \alpha}(s_i)
$$

<br>

### **Bước 4: Tính cận dưới và cận trên cho điểm test**

Với xác suất dự đoán :

$$
\text{Lower} = \max(0, \hat{p}_{\text{test}} - q_\alpha)
$$

$$
\text{Upper} = \min(1, \hat{p}_{\text{test}} + q_\alpha)
$$

Điều này đảm bảo rằng, với xác suất ít nhất là \( 1 - \alpha \), giá trị thực của xác suất sẽ nằm trong khoảng trên.

<br>

### **Ví dụ minh họa**

Giả sử:
- \( alpha = 0.05 \)
- \( q_{0.05} = 0.15 \)
- \( q_test = 0.88 \)

Tính:

$$
\text{Lower} = 0.88 - 0.15 = 0.73
$$

$$
\text{Upper} = clip(1.00, 0.88 + 0.15) = 1.00
$$

→ Prediction interval: [0.73, 1.00]

<br>

### **Kết luận**

$$
\boxed{
\begin{aligned}
\text{Lower} &= \max(0, \hat{p}_{\text{test}} - q_\alpha) \\
\text{Upper} &= \min(1, \hat{p}_{\text{test}} + q_\alpha)
\end{aligned}
}
$$


In [ ]:
# FIX: Định nghĩa alpha và tính prob_lower/upper trước khi dùng
# Dùng ALPHA_MAPIE đã khai báo ở đầu notebook (0.05 → 95% confidence)

_, y_pred_set_single = mapie_clf.predict(X_test, alpha=ALPHA_MAPIE)

# Tính probability interval thủ công theo công thức LAC
prob_test = best_model.predict_proba(X_test)[:, 1]

# Conformity scores trên calibration set
prob_calib = best_model.predict_proba(X_calib)[:, 1]
conf_scores = np.where(
    y_calib == 1,
    1 - prob_calib,   # class 1: score = 1 - p
    prob_calib        # class 0: score = p
)
q_alpha = np.quantile(conf_scores, 1 - ALPHA_MAPIE)

prob_lower = np.maximum(0, prob_test - q_alpha)
prob_upper = np.minimum(1, prob_test + q_alpha)

interval_df = pd.DataFrame({
    'y_test':       y_test.values,
    'Prob Lower':   np.round(prob_lower, 4),
    'Prob Class 1': np.round(prob_test, 4),
    'Prob Upper':   np.round(prob_upper, 4),
})

# Thêm Prediction Set
interval_df['Set'] = [
    set(np.where(row)[0]) for row in y_pred_set_single[:, :, 0]
]

print(f"Quantile q_{{alpha}} = {q_alpha:.4f}  (alpha={ALPHA_MAPIE}, confidence={1-ALPHA_MAPIE:.0%})")
display(interval_df.head(20))
